### Bước 1: Thiết lập môi trường và Cài đặt thư viện

Cài đặt các gói thư viện cần thiết cho việc xử lý ngôn ngữ tự nhiên tiếng Việt và huấn luyện mô hình Transformer:

    - transformers[torch]: Thư viện chính để làm việc với các mô hình BERT.

    - datasets: Hỗ trợ quản lý tập dữ liệu.

    - scikit-learn: Cung cấp các công cụ chia tập dữ liệu và tính toán chỉ số đánh giá.

    - pyvi: Thư viện tách từ (word segmentation) chuyên dụng cho tiếng Việt.

Khai báo các thư viện Python dùng cho xử lý dữ liệu, đại số tuyến tính và các thành phần cấu thành mô hình:

    - pandas, numpy: Xử lý bảng dữ liệu và mảng.
    
    - torch: Thư viện học sâu PyTorch.
    
    - transformers: Tải Tokenizer, Model và cấu hình huấn luyện.
    
    - sklearn.model_selection: Chia tập dữ liệu Train/Validation

### Bước 3: Giải nén dữ liệu và Tiền xử lý

Giải nén tệp midtermNLP01.zip vào thư mục data_nlp. Sau đó, sử dụng os.walk để quét và xác định đường dẫn chính xác của tệp train.csv và test.csv trước khi nạp vào DataFrame.

Định nghĩa bản đồ chủ đề (Topic Map): Ánh xạ các mã số chủ đề (0, 1, 2, 3) sang văn bản mô tả (Giảng viên, Chương trình đào tạo,...).

Hàm clean_and_combine:
    
    - Chuyển văn bản về chữ thường.
    
    - Sử dụng ViTokenizer để tách từ tiếng Việt (ví dụ: "sinh viên" thành "sinh_viên").
    
    - Kết hợp thông tin Chủ đề vào đầu câu phản hồi bằng token đặc biệt [SEP] (ví dụ: Chủ_đề [SEP] Nội_dung_câu) để mô hình BERT hiểu ngữ cảnh.

### Bước 4: Chuẩn bị Dataset và Tokenization
Khởi tạo Tokenizer: Sử dụng vinai/phobert-base-v2 để chuyển văn bản thành các dãy số (ids) mà máy tính hiểu được.

Chia tập dữ liệu: Chia tập huấn luyện (Train) và tập kiểm thử nội bộ (Validation) theo tỷ lệ 90/10, có sử dụng stratify để đảm bảo cân bằng tỷ lệ nhãn cảm xúc.

Lớp FeedbackDataset: Xây dựng lớp kế thừa từ * torch.utils.data.Dataset * để quản lý việc mã hóa văn bản (encoding) và nhãn (labels) trong quá trình huấn luyện.

### Bước 5: Cấu hình và Huấn luyện mô hình
Khởi tạo mô hình: Tải mô hình PhoBERT-base-v2 với 3 nhãn đầu ra (Tiêu cực, Trung tính, Tích cực).

Thiết lập TrainingArguments:

num_train_epochs=5: Số vòng lặp huấn luyện.

learning_rate=2e-5: Tốc độ học tối ưu cho các dòng BERT.

load_best_model_at_end=True: Tự động tải lại phiên bản mô hình có chỉ số tốt nhất sau khi kết thúc.

Sử dụng chỉ số macro_f1 để đánh giá hiệu quả mô hình.

Dừng sớm (EarlyStoppingCallback): Nếu chỉ số macro_f1 không cải thiện sau 3 lần epoch (patience=3) thì ngừng huấn luyện.

### Bước 6: Dự đoán và Xuất kết quả
Dự đoán: Sử dụng tập Test đã qua tiền xử lý để mô hình đưa ra nhãn dự đoán.

Xử lý đầu ra: Sử dụng np.argmax để chọn nhãn có xác suất cao nhất từ kết quả (logits) của mô hình.

Xuất file: Tạo DataFrame chứa cột id và sentiment, lưu thành tệp submission.csv và tải về máy để nộp bài.


In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers[torch] datasets scikit-learn pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.2 MB/s eta 0:00:00


In [ ]:
import os
import re
import zipfile
import pandas as pd
import numpy as np
import torch
from pyvi import ViTokenizer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

In [ ]:
zip_file_name = 'midtermNLP01.zip'
extract_path = 'data_nlp'

if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"--- Đã giải nén file {zip_file_name} thành công ---")
else:
    print(f"!!! Không tìm thấy file {zip_file_name}. Hãy đảm bảo bạn đã upload file lên thư mục gốc của Colab.")

train_path = None
test_path = None
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file == 'train.csv': train_path = os.path.join(root, file)
        if file == 'test.csv': test_path = os.path.join(root, file)

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

--- Đã giải nén file midtermNLP01.zip thành công ---


In [ ]:
topic_map = { 0: "Giảng_viên", 1: "Chương_trình_đào_tạo", 2: "Cơ_sở_vật_chất", 3: "Khác" }

def clean_and_combine(row):
    sentence = ViTokenizer.tokenize(str(row['sentence']).lower())
    topic_id = row.get('topic', 3) # Mặc định là 'Khác' nếu không có cột topic
    topic_name = topic_map.get(topic_id, "Khác")
    # Kết hợp Topic và Nội dung bằng token [SEP] của BERT. Cấu trúc: "Chủ_đề [SEP] câu văn đã tách từ"
    return f"{topic_name} [SEP] {sentence}"

df_train['final_text'] = df_train.apply(clean_and_combine, axis=1)
df_test['final_text'] = df_test.apply(clean_and_combine, axis=1)

In [ ]:
print(df_train.head(10))
print(df_train.info())

   id                                           sentence  topic  sentiment  \
0   0                  thầy đảm bảo tốt về giờ lên lớp .      0          1   
1   1                            tinh thần , kiến thức .      3          1   
2   2  nhiều học sinh không hài lòng với cách sắp lịc...      0          0   
3   3  thầy rất nhiệt tình , giảng rất hay , nhiều ví...      0          2   
4   4              nhiệt tình , vui vẻ , hướng dẫn tốt .      0          2   
5   5  đề nghị có câu trả lời cụ thể khi sinh viên hỏ...      0          0   
6   6                        thầy nhiệt tình dễ thương .      0          2   
7   7                          thầy cho chép nhiều quá .      0          0   
8   8  cô dạy nhiệt tình , kiến thức sâu và rộng , li...      0          2   
9   9  cô sửa nhiều bài tập giúp sinh viên dễ tiếp th...      0          2   

                                          final_text  
0  Giảng_viên [SEP] thầy đảm_bảo tốt về giờ lên_l...  
1                 Khác [SEP] ti

In [ ]:
print(df_test.head(10))
print(df_test.info())

   id                                           sentence  \
0   0  kiến thức môn anh văn được đáp ứng nhưng không...   
1   1                            không nghiêm khắc lắm .   
2   2           cô giảng dễ hiểu , đủ nội dung bài học .   
3   3  thầy khá nhiệt huyết , vui vẻ , hệ thống kiến ...   
4   4  cần nâng cao cơ sở vật chất và phòng học vì th...   
5   5         giáo viên giảng dạy nhiệt tình , dễ hiểu .   
6   6  cải thiện chất lượng các phòng máy và kết nối ...   
7   7                                 thầy dạy rất vui .   
8   8             cần có tài liệu để đọc thêm , tự học .   
9   9  thời gian hoàn thành đồ án quá gấp rút , khi c...   

                                          final_text  
0  Khác [SEP] kiến_thức môn anh văn được đáp_ứng ...  
1                 Khác [SEP] không nghiêm_khắc lắm .  
2  Khác [SEP] cô giảng dễ hiểu , đủ nội_dung bài_...  
3  Khác [SEP] thầy khá nhiệt_huyết , vui_vẻ , hệ_...  
4  Khác [SEP] cần nâng cao cơ_sở vật_chất và phòn...  
5  Khác [

In [ ]:
model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class FeedbackDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train['final_text'].tolist(),
    df_train['sentiment'].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=df_train['sentiment']
)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=160)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=160)

train_dataset = FeedbackDataset(train_encodings, train_labels)
val_dataset = FeedbackDataset(val_encodings, val_labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    return {"macro_f1": macro_f1}

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,              # Chạy 5 epoch để đạt độ chín
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,              # Tốc độ học tối ưu cho PhoBERT
    weight_decay=0.05,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,     # Tự động lấy bản tốt nhất theo F1
    metric_for_best_model="macro_f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Dừng sớm nếu không tiến triển
)

trainer.train()

print("--- Đang dự đoán trên tập Test ---")
test_encodings = tokenizer(df_test['final_text'].tolist(), truncation=True, padding=True, max_length=160)
test_dataset = FeedbackDataset(test_encodings)
raw_preds = trainer.predict(test_dataset)
final_predictions = np.argmax(raw_preds.predictions, axis=-1)

submission = pd.DataFrame({
    'id': df_test['id'],
    'sentiment': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("File 'submission.csv' đã được tạo thành công.")

from google.colab import files
files.download('submission.csv')

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
